# Analitika Sains Data Sepak Bola: AFC Asian Cup 2027
## Proyeksi Probabilitas 24 Negara Peserta & Peluang Timnas Indonesia Melaju ke Babak 8 Besar

Notebook ini menyajikan analisis kuantitatif komprehensif berbasis data pertandingan 4 tahun terakhir (2023–2026), evolusi nilai pasar skuad pemain diaspora, serta simulasi turnamen 100.000 iterasi Monte Carlo.

### Komposisi Grup Resmi 2027:
- **Grup A**: Arab Saudi, Kuwait, Oman, Palestina
- **Grup B**: Uzbekistan, Bahrain, Korea Utara, Yordania
- **Grup C**: Iran, Suriah, Kirgizstan, China
- **Grup D**: Australia, Tajikistan, Irak, Singapura
- **Grup E**: Korea Selatan, Uni Emirat Arab, Vietnam, Yaman
- **Grup F**: Jepang, Qatar, Thailand, Indonesia

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi estetika grafik bertema gelap
plt.style.use('dark_background')
DARK_BG = '#0d1117'
PANEL_BG = '#161b22'
BORDER_COLOR = '#30363d'
TEXT_COLOR = '#f0f6fc'
TEXT_MUTED = '#8b949e'
IDN_COLOR = '#e63946'
GOLD_COLOR = '#fbbf24'
CYAN_COLOR = '#38bdf8'

# Memuat berkas data
BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
CSV_PATH = os.path.join(BASE_DIR, 'data', 'asian_cup_predictions.csv')
JSON_PATH = os.path.join(BASE_DIR, 'data', 'indonesia_4yr_match_analytics.json')

df_preds = pd.read_csv(CSV_PATH)
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    analytics_idn = json.load(f)

print('Data prediksi 24 tim dan analitika Indonesia berhasil dimuat!')
df_preds.head(10)

### 1. Peta Peluang Lolos ke Babak 8 Besar (Perempat Final)
Membandingkan 12 negara dengan probabilitas perempat final tertinggi di Asia.

In [ ]:
top_qf = df_preds.sort_values(by='Reach_Quarter_Final_Prob(%)', ascending=False).head(12)

fig, ax = plt.subplots(figsize=(12, 6), facecolor=DARK_BG)
ax.set_facecolor(PANEL_BG)

colors = [IDN_COLOR if t == 'Indonesia' else CYAN_COLOR for t in top_qf['Team']]
bars = ax.barh(top_qf['Team'][::-1], top_qf['Reach_Quarter_Final_Prob(%)'][::-1], color=colors[::-1], edgecolor=BORDER_COLOR)

ax.set_title('PROBABILITAS LOLOS BABAK 8 BESAR (PEREMPAT FINAL) PIALA ASIA 2027', fontsize=13, weight='bold', color=TEXT_COLOR, pad=15)
ax.set_xlabel('Peluang Lolos (%)', fontsize=11, color=TEXT_COLOR)
ax.grid(color=BORDER_COLOR, linestyle='--', alpha=0.5, axis='x')

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.8, bar.get_y() + bar.get_height()/2, f'{w:.1f}%', va='center', ha='left', fontsize=10, weight='bold', color=TEXT_COLOR)

plt.tight_layout()
plt.show()

### 2. Evolusi Nilai Pasar Skuad Timnas Indonesia (2023–2026)
Lonjakan nilai pasar dari €5,85M menjadi €36,5M yang mendongkrak daya saing Garuda ke peringkat 6 Asia.

In [ ]:
yearly = analytics_idn['squad_market_value_evolution']['yearly_trajectory']
years = [d['year'].split(' (')[0] for d in yearly]
vals = [d['market_value_eur'] / 1e6 for d in yearly]
ranks = [d['rank_in_asia'] for d in yearly]

fig, ax = plt.subplots(figsize=(10, 5), facecolor=DARK_BG)
ax.set_facecolor(PANEL_BG)

ax.plot(years, vals, marker='o', color=IDN_COLOR, linewidth=3, markersize=8, label='Nilai Pasar Skuad (€ Juta)')
ax.fill_between(years, vals, color=IDN_COLOR, alpha=0.2)

for x, y, r in zip(years, vals, ranks):
    ax.text(x, y + 1.2, f'€{y:.1f}M\n(Peringkat #{r} Asia)', ha='center', va='bottom', fontsize=9.5, weight='bold', color=TEXT_COLOR)

ax.set_title('TRANSFORMASI NILAI PASAR SKUAD TIMNAS INDONESIA (2023–2026)', fontsize=12, weight='bold', color=TEXT_COLOR, pad=15)
ax.set_ylabel('Nilai Pasar Skuad (€ Juta)', fontsize=10, color=TEXT_COLOR)
ax.set_ylim(0, 45)
ax.grid(color=BORDER_COLOR, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### 3. Bedah 3 Skenario Timnas Indonesia Menuju Babak 8 Besar dari Grup F
Grup F: Jepang, Qatar, Thailand, Indonesia.

In [ ]:
scenarios = analytics_idn['road_to_quarter_final_8_besar']['scenario_breakdown_to_8_besar']
scen_df = pd.DataFrame(scenarios)
scen_df[['scenario', 'likelihood_to_occur', 'opponent_in_r16', 'r16_win_probability', 'verdict']]